# Predicting Stellar Class (Kaggle Playground Series - Season 6, Episode 6)

## 元notebook: Galaxy Population XGB

- **原著者**: Marília Prata ([mpwolke](https://www.kaggle.com/mpwolke))
- **元notebookへのリンク**: https://www.kaggle.com/code/mpwolke/galaxy-population-xgb
- **評価**: 55 upvotes / Silver medal

### 概要

SDSS17（Sloan Digital Sky Survey）のスペクトルデータから、天体を `GALAXY`（銀河）・`QSO`（クエーサー）・`STAR`（恒星）の3クラスに分類するコンペ。この notebook は、天文学の基礎知識（赤方偏移、恒星のスペクトル型、銀河のレッドシーケンス/ブルークラウド/グリーンバレー）を丁寧に解説しながらデータを探索し、最終的にシンプルな `XGBClassifier` で約96.8%の精度を達成する、という構成になっている。モデル自体は複雑ではなく、**ドメイン知識を理解した上で素直にEDA→前処理→学習を行う**という基本の型を学ぶのに向いている。

### 断り書き

これは学習目的の解説付き写しです。元のコードセルの中身は変更していませんが、このノートブックは未実行のため、グラフや表などの出力は含まれていません。

## 環境準備（ライブラリのインポート）

**What**: `numpy`・`pandas`・`matplotlib`・`seaborn`・`plotly` など、データ分析でよく使うライブラリをまとめて読み込む。あわせて `/kaggle/input` 以下にあるファイル一覧を表示し、警告表示をオフにしている。

**Why**: Kaggle Notebookのテンプレートに含まれる定型的な準備セル。どんなファイルが使えるかを最初に確認しておくことで、この後の `read_csv` のパス指定ミスを防げる。

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

#Two lines Required to Plot Plotly
import plotly.io as pio
pio.renderers.default = 'iframe'

import plotly.graph_objs as go
import plotly.offline as py
import plotly.express as px

#Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## データの読み込みと特徴量の意味

**What**: 学習データ（`train.csv`）・テストデータ（`test.csv`）・提出用サンプル（`sample_submission.csv`）をpandasで読み込む。

**Why**: 機械学習コンペでは、まずデータを読み込んで手元に置くのが最初のステップ。今回の特徴量には天文学特有の用語が多いので、あらかじめ意味を押さえておくと後の分析が理解しやすくなる。

- `alpha` / `delta`: 天体の位置を表す座標（地球でいう経度・緯度にあたる「赤経・赤緯」）
- `u, g, r, i, z`: 5種類のフィルター（紫外線〜近赤外線）を通した際の明るさ（測光システムの値）
- `redshift`（赤方偏移）: 天体が地球からどれだけ速く遠ざかっているか＝おおよその距離の指標

（参考: [SDSS17データセット詳細](https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17/data)）

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv')

## データ末尾の確認

**What**: `train.tail(3)` で学習データの最後の3行を表示する。

**Why**: `head()` だけでなく `tail()` も見ることで、データの並び順に偏りがないか、末尾に想定外の値が混ざっていないかをざっと確認できる。特に `id` 列が連番かどうかもここで分かる。

In [ ]:
train.tail(3)

## データ型・欠損値の確認

**What**: `train.info()` で各列のデータ型と非null件数を確認する。

**Why**: 数値列が文字列として読み込まれていないか、欠損値（NaN）がどの列にどれくらいあるかを最初に把握しておくことは、前処理の方針を決める上で欠かせないチェック。

In [ ]:
train.info()

## 恒星のスペクトル型について

**What**: `train['spectral_type'].value_counts()` で `spectral_type` 列の値ごとの件数を数える。

**Why**: `spectral_type` は恒星の表面温度やスペクトル線の特徴によって恒星を分類したもの（ハーバード分類: O, B, A, F, G, K, M の順で高温→低温）。カテゴリごとにどれくらいデータがあるかを事前に知っておくと、後で分布を可視化したときの見方が変わる。
（参考: [ハーバード分類の解説](https://astro.unl.edu/naap/hr/hr_background1.html)）

In [ ]:
train['spectral_type'].value_counts()

## 銀河のレッドシーケンス／ブルークラウド／グリーンバレーについて

これは独立したコードセルはないが、次のグラフを理解するための背景知識。

- **レッドシーケンス**: 古く低温な星が多く、星形成がほぼ止まった（「クエンチ」された）楕円銀河などのグループ。色は赤〜黄色っぽい。
- **ブルークラウド**: 若く高温な星が多く活発に星を作っている渦巻銀河・不規則銀河のグループ。色は青っぽい。
- **グリーンバレー**: レッドシーケンスとブルークラウドの間にある、星形成が弱まりつつある過渡的なグループ。

（出典: Eales et al., *"The causes of the red sequence, the blue cloud, the green valley, and the green mountain"*, MNRAS, 2018. https://doi.org/10.1093/mnras/sty2220）

**What / Why**: 次のセルでは `spectral_type`・`galaxy_population`（Red Sequence / Blue Cloud）・`class`（GALAXY/QSO/STAR）という3つのカテゴリ変数の分布を、棒グラフで一気に可視化する。カテゴリの偏り（不均衡）を最初に見ておくことで、後のクラス分類での注意点（多数派クラスに引っ張られやすい、など）が見えてくる。

In [ ]:
#By H-Z-Ning  https://www.kaggle.com/code/hzning/top-10-solution-0-97525-esay-is-all-you

categorical_columns = ["spectral_type", "galaxy_population", "class"]

plt.figure(figsize=(14, 12))
for i, column in enumerate(categorical_columns, 1):
    plt.subplot(3, 3, i)
    sns.countplot(x=column, data=train, palette='Set2')
    plt.title(f'Distribution of {column}')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 基本統計量の確認

**What**: `train.describe(include='all')` を `display()` で表示し、数値列・カテゴリ列の両方について件数・平均・最小最大・ユニーク数などをまとめて確認する。

**Why**: 各特徴量のスケール（数値の大きさの違い）や外れ値の有無をざっと掴んでおくと、モデルの挙動を理解しやすくなる。`include='all'` にすることでカテゴリ列（`spectral_type` など）の統計も一緒に見られる。

In [ ]:
display(train, train.describe(include='all'))

## 相関ヒートマップ

**What**: 数値列同士の相関係数をヒートマップで可視化する。

**Why**: `u, g, r, i, z`（明るさのフィルター値）同士がどれくらい強く相関しているかを見ることで、特徴量に情報の重複（多重共線性）がないかを確認できる。今回は `r, i, z` あたりが強く相関している様子が読み取れる想定で、この後のモデリングでの特徴量選択の参考になる。

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(train.select_dtypes(include='number').corr(), annot=True, cmap='summer');

## 目的変数（ターゲット）の分布確認

**What**: `class` 列（GALAXY / QSO / STAR）の件数を棒グラフで可視化する。

**Why**: 3クラス分類ではクラスごとの件数のバランスが重要。GALAXYが多数派でSTARが少数派、といった不均衡があると、モデルが多数派に偏った予測をしやすくなるため、精度指標の見方や後の学習設計（重み付けなど）を考える際の材料になる。

In [ ]:
# Plot target distribution
plt.figure(figsize=(6,4))
sns.countplot(x='class', data=train, palette='summer')
plt.title('Stellar Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

## 目的変数と不要な列の分離

**What**: 予測対象である `class` 列を `y` として取り出し、`X` からは `class`・`id`（識別子で予測に無関係）・`spectral_type`・`galaxy_population`（このあとエンコードせずにそのまま使うと文字列のままモデルに渡ってしまう列）を除外する。

**Why**: 機械学習モデルの多くは数値の特徴量しか扱えないため、学習に使う特徴量（X）と予測対象（y）を明確に分け、文字列のままの列を一旦除いておく必要がある。

In [ ]:
# Get prediction column
y = train['class']
# Remove class column and other unneeded columns
X = train.drop(['class', 'id', 'spectral_type', 'galaxy_population' ], axis=1, inplace=False)

## QSO（クエーサー）とは

**What/Why**: 次のセルでラベルを数値化する前に、`class` の一つである `QSO` について触れておく。QSO は Quasi-Stellar Object（準恒星状天体）の略で、一般には「クエーサー」と呼ばれる。超巨大ブラックホールに物質が落ち込む際に放つ、非常に明るい銀河の中心核のこと。遠方にあるため、望遠鏡では恒星のような点光源に見えるのが名前の由来。
（参考: https://www.aanda.org/articles/aa/full_html/2013/12/aa22196-13/aa22196-13.html）

**What**: `y` の文字列ラベル（GALAXY/QSO/STAR）を、モデルが扱える整数（0/1/2）に変換する。

**Why**: XGBoostなどの分類器は数値のクラスラベルを前提とすることが多いため、文字列→整数のマッピングが必要。

In [ ]:
# Make y numerical so it works with model
y = y.replace({'GALAXY': 0, 'QSO': 1, 'STAR': 2})

## 学習用・検証用データへの分割

**What**: `train_test_split` でデータを学習用（75%）と検証用（25%）に分割する。

**Why**: モデルが未知のデータにどれくらい汎化できるかを確認するため、学習に使わないデータ（検証用）を確保しておく必要がある。`random_state=42` を固定することで、結果を再現できるようにしている。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=42)

len(X_train), len(y_train), len(X_test), len(y_test)

## XGBoost分類器の定義

**What**: `XGBClassifier` を、木の本数 `n_estimators=1000`、`early_stopping_rounds=5`（検証スコアが5ラウンド改善しなければ打ち切り）という設定で用意する。

**Why**: XGBoost（Extreme Gradient Boosting）は決定木を順番に追加していき、前の木の誤差を次の木が補正していく手法（勾配ブースティング）。テーブルデータのコンペで非常によく使われる定番モデル。early stoppingを使うことで、過学習を防ぎつつ木の本数を自動的に適切な数に抑えられる。

In [ ]:
#By Edwardia Fosah https://www.kaggle.com/code/edwardiafosah/stellar-classification

import xgboost
from xgboost import XGBClassifier

xgb_classifier = XGBClassifier(n_estimators=1000, early_stopping_rounds=5)

## モデルの学習

**What**: 学習データで `fit()` し、`eval_set` に検証データを渡すことで、学習中に検証スコアを監視しながら early stopping を機能させる。

**Why**: `eval_set` を指定しないと early stopping は動作しない。学習と検証を同時に見ることで、過学習が始まったタイミングで自動的に学習を止められる。

In [ ]:
#By Edwardia Fosah https://www.kaggle.com/code/edwardiafosah/stellar-classification

xgb_classifier.fit(X_train, y_train,
                    eval_set=[(X_test, y_test)])

## 検証データでの予測

**What**: 学習済みモデルで検証用データ（`X_test`）のクラスを予測する。

**Why**: 精度を計算する前段階として、実際にモデルがどのクラス（0/1/2）を出力するかを確認する。

In [ ]:
#By Edwardia Fosah https://www.kaggle.com/code/edwardiafosah/stellar-classification

preds = xgb_classifier.predict(X_test)
preds

## 精度（Accuracy）の計算

**What**: 予測値 `preds` と正解ラベル `y_test` を比較し、正解率を計算する。

**Why**: シンプルな正解率（Accuracy）は分類モデルの最も基本的な評価指標。元notebookではこのシンプルなXGBoostベースラインで約96.8%の精度が出ている。

In [ ]:
#By Edwardia Fosah https://www.kaggle.com/code/edwardiafosah/stellar-classification

print(f"Accuracy: {sum(preds == y_test) / len(y_test)}")

## Acknowledgements（謝辞・参考）

元notebookの著者 Marília Prata (mpwolke) が参考にした notebook:

- Edwardia Fosah: https://www.kaggle.com/code/edwardiafosah/stellar-classification
- H-Z-Ning: https://www.kaggle.com/code/hzning/top-10-solution-0-97525-esay-is-all-you

このノートブックはApache 2.0ライセンスの下で公開されている。